# Cat vs Dog Image Classification using CNN

**Goal:** classify an image as a cat or a dog using a Convolutional Neural Network (CNN).

This notebook is designed for **Google Colab or Kaggle**. It looks for a `PetImages/Cat` and `PetImages/Dog` directory automatically. Keep the image dataset outside GitHub because it is large.

> Dataset note: the Microsoft Cats vs Dogs dataset can contain a small number of damaged images. The pipeline skips decoding errors where supported. Results must be generated by running this notebook; no accuracy is claimed in advance.

## 1. Imports and configuration

In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

print("TensorFlow:", tf.__version__)
SEED = 42
IMG_SIZE = (150, 150)
BATCH_SIZE = 32
EPOCHS = 10
tf.keras.utils.set_random_seed(SEED)

## 2. Find the dataset

In [ ]:
# Expected structure:
# PetImages/
#   Cat/  (cat images)
#   Dog/  (dog images)
#
# Kaggle input paths and common Colab paths are checked automatically.
candidate_roots = [
    Path("/kaggle/input/microsoft-catsvsdogs-dataset/PetImages"),
    Path("/kaggle/input/cats-vs-dogs/PetImages"),
    Path("/kaggle/input/cats-vs-dogs-dataset/PetImages"),
    Path("/content/PetImages"),
    Path("/content/Cats-vs-Dogs/PetImages"),
    Path("/content/cats-vs-dogs/PetImages"),
]

DATA_DIR = next((p for p in candidate_roots if (p / "Cat").is_dir() and (p / "Dog").is_dir()), None)

if DATA_DIR is None:
    # Last-resort search under common dataset roots.
    search_roots = [Path("/kaggle/input"), Path("/content")]
    for base in search_roots:
        if base.exists():
            matches = list(base.glob("**/PetImages"))
            DATA_DIR = next((p for p in matches if (p / "Cat").is_dir() and (p / "Dog").is_dir()), None)
            if DATA_DIR is not None:
                break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find PetImages/Cat and PetImages/Dog. "
        "In Kaggle, attach the Cats-vs-Dogs dataset as an input. "
        "In Colab, upload/extract the dataset so /content/PetImages/Cat and /content/PetImages/Dog exist."
    )

print("Dataset directory:", DATA_DIR)
print("Cat files:", len(list((DATA_DIR / "Cat").iterdir())))
print("Dog files:", len(list((DATA_DIR / "Dog").iterdir())))

## 3. Load and split the images

In [ ]:
# Use an 80/20 training/validation split. Validation results are not a separate test-set result.
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels="inferred",
    label_mode="int",
    class_names=["Cat", "Dog"],
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels="inferred",
    label_mode="int",
    class_names=["Cat", "Dog"],
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

class_names = train_ds.class_names
print("Class order:", class_names)

# Normalize pixel values to [0, 1].
normalization = tf.keras.layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization(x), y), num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.map(lambda x, y: (normalization(x), y), num_parallel_calls=tf.data.AUTOTUNE)

# Prefetch improves input-pipeline throughput.
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

## 4. Preview sample images

In [ ]:
images, labels = next(iter(train_ds))
plt.figure(figsize=(10, 8))
for i in range(min(9, len(images))):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy())
    plt.title(class_names[int(labels[i].numpy())])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 5. Build the CNN

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
], name="data_augmentation")

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(*IMG_SIZE, 3)),
    data_augmentation,
    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(2, activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## 6. Train the model

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "best_cat_dog_model.keras", monitor="val_loss", save_best_only=True
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

## 7. Plot training history

In [ ]:
history_dict = history.history
epochs_range = range(1, len(history_dict["accuracy"]) + 1)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history_dict["accuracy"], label="Training accuracy")
plt.plot(epochs_range, history_dict["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Accuracy")

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history_dict["loss"], label="Training loss")
plt.plot(epochs_range, history_dict["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Loss")
plt.tight_layout()
plt.show()

## 8. Evaluate on the validation split

In [ ]:
val_loss, val_accuracy = model.evaluate(val_ds, verbose=1)
print(f"Validation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy:.2%}")
print("Note: this is validation performance, not an independent test-set score.")

## 9. Inspect predictions

In [ ]:
images, labels = next(iter(val_ds))
probabilities = model.predict(images, verbose=0)
predicted_ids = np.argmax(probabilities, axis=1)

plt.figure(figsize=(12, 9))
for i in range(min(12, len(images))):
    true_name = class_names[int(labels[i].numpy())]
    pred_name = class_names[int(predicted_ids[i])]
    confidence = float(np.max(probabilities[i]))
    ax = plt.subplot(3, 4, i + 1)
    plt.imshow(images[i].numpy())
    color = "green" if true_name == pred_name else "red"
    plt.title(f"Pred: {pred_name} ({confidence:.0%})\nTrue: {true_name}", color=color)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 10. Save the trained model

In [ ]:
model.save("cat_dog_cnn.keras")
print("Saved model to cat_dog_cnn.keras")

## Conclusion

This notebook builds and trains a CNN for binary cat/dog image classification, uses augmentation during training, plots learning curves, evaluates validation performance, and visualizes predictions.

Before publishing results, run the notebook and replace this note with your actual validation metrics. For a stronger evaluation, create a separate held-out test set and report its results independently.